In [ ]:
pip install --upgrade google-api-python-client

In [ ]:
from googleapiclient.discovery import build

In [ ]:
import json

In [ ]:
API_KEY="YOUR KEY"

In [ ]:
youtube=build('youtube','v3',developerKey=API_KEY)

In [ ]:
request=youtube.videos().list(part="snippet,statistics",chart="mostPopular",regionCode="IN",maxResults=500)

In [ ]:
response=request.execute()

In [ ]:
print(response)

In [ ]:
videos=[]

In [ ]:
for item in response['items']:
    videos.append({
    "title":item['snippet']['title'],
    "channel":item['snippet']['channelTitle'],
    "views":item['statistics'].get('viewCount',0),
    "likes":item['statistics'].get('likeCount',0)})
    

In [ ]:
import pandas as pd

In [ ]:
with open("youtube_raw.json","w")as f:
    json.dump(videos,f,indent=4)

In [ ]:
df=pd.DataFrame(videos)

In [ ]:
df.head()

In [ ]:
df['views']=pd.to_numeric(df['views'],errors='coerce')

In [ ]:
df['likes']=pd.to_numeric(df['likes'],errors='coerce')

In [ ]:
df.isnull().sum()

In [ ]:
df.describe()

In [ ]:
pip install pandasql

In [ ]:
from pandasql import sqldf
import pandas as pd
pysql=lambda q: sqldf(q,globals())
result= pysql("""
SELECT title,channel,views
FROM df
ORDER BY likes DESC 
LIMIT 10""")


In [ ]:
result_likes=pysql("""
SELECT title,channel,likes
FROM df
ORDER BY likes
LIMIT 10""")

In [ ]:
result

In [ ]:
result_both=pysql("""
SELECT title,channel,likes,views
FROM df
ORDER BY likes DESC,views DESC
LIMIT 10""")

In [ ]:
result_both

In [ ]:
result_engagement=pysql("""
SELECT title,channel,ROUND((likes*100/views),2) as public_engagement
FROM df
ORDER BY public_engagement DESC
LIMIT 10""")


In [ ]:
result_engagement

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
channel_views=df.groupby('channel')['views'].sum().sort_values(ascending=False).head(10)
plt.figure(figsize=(12,6))
plt.pie(channel_views,labels=channel_views.index,autopct='%1.1f%%')
plt.title("top 10 channel by views")
plt.savefig("youtube_piechart.png")
plt.show()

import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
top_views=df.sort_values('views',ascending=False).head(10)
plt.figure(figsize=(12,6))
plt.barh(top_views['title'].str[:30],top_views['views'])
plt.title("top 10 videos by views")
plt.xlabel("views")
plt.ylabel("video title")
plt.tight_layout()
plt.savefig("youtube_barchart")
plt.show()